# Tune Thresholds

Chooses the thresholds that `Segment_Specimens.ipynb` uses to tell beetles from look-alikes (bees, wasps...) and from things that aren't insects at all, and **compares the two scoring modes** so you can see whether the simpler one is good enough.

**The idea.** Step 3b of `Segment_Specimens.ipynb` scores a random sample of boxes **once**, with both scoring modes. Every combination of thresholds can then be tested on those saved scores instantly, with no model running. You supply the truth for a sample of boxes by sorting their pictures into three folders; the notebook counts the mistakes for every combination.

**Joint mode** (one question over all categories; try this first):

| the box is... | when |
|---|---|
| a **look-alike** → rejected | look-alike score ≥ `LOOKALIKE_MIN` **and** higher than the beetle score |
| a **beetle** → kept | otherwise, beetle score ≥ `BEETLE_MIN` |
| **not an insect** → rejected | otherwise |

**Separate mode** (one yes/no question per category; try it if joint isn't good enough):

| insect score | beetle score ≥ `SEP_BEETLE_MIN` | look-alike score ≥ `SEP_LOOKALIKE_MIN` | decision |
|---|---|---|---|
| below `SEP_INSECT_MIN` | | | **not an insect** → rejected |
| ok | yes | no | **beetle** → kept |
| ok | yes | yes | **review** → set aside for you to check |
| ok | no | yes | **look-alike** → rejected |
| ok | no | no | **neither** → rejected |

**Steps**
1. Settings. 2. Load the scores. 3. Make a sample of crops to label, then **sort them by hand** into the `beetle`, `lookalike` and `other` folders (the one manual step). 4. Read your labels. 5. Try every threshold combination in both modes, and pick the best. 6. Sort *all* scored crops into folders for the best combinations, to look at the mistakes. 7. Check the rejection rate in each family (e.g. Staphylinidae).

**Kernel:** any environment with numpy, pandas, matplotlib and Pillow, such as the **specimen** one. No models are needed.

**Words used.** Beetle is the "positive" class. *True beetles rejected* are beetles wrongly thrown out (false negatives: lost training images). *Non-beetles passed* are bees, labels etc. wrongly kept (false positives: contamination). **Sensitivity** = share of true beetles kept. **Specificity** = share of non-beetles rejected. **Review** boxes (separate mode) are assumed to be sorted correctly by you afterwards, so they don't count as mistakes, but they cost you work, so the number of them is capped (`MAX_REVIEW_PERCENT`). The tables also show `strict_sensitivity`, which counts true beetles that ended up in review as lost.

## Step 1: Settings

- **`TUNING_DIR`**: the `tuning` folder that Step 3b made, inside `Data/<name>_segmented_extras/`. `None` opens a pop-up to choose it.
- **`LABEL_SAMPLE_SIZE`**: how many crops you will label by hand. More is better, especially for the rarer look-alikes. **Aim for at least 30 in each of `beetle` and `lookalike`**, or the estimates for that group are rough.
- **Threshold lists**: every combination is tested. Joint mode tries pairs; separate mode tries triples. The defaults include the lab's values (beetle 0.8, 0.85, 0.9; look-alike 0.5, 0.6, 0.7).
- **`TARGET_SENSITIVITY`**, **`TARGET_SPECIFICITY`**: what "good enough" means. The picks below keep at least `TARGET_SENSITIVITY` of true beetles and then maximise specificity; the verdict says whether `TARGET_SPECIFICITY` was reached.
- **`MAX_REVIEW_PERCENT`**: separate mode only. Combinations that send more than this percentage of boxes to review are not eligible.
- **`SORT_COMBINATIONS`**: combinations to sort into folders in Step 6: `("joint", beetle_min, lookalike_min)` or `("separate", insect_min, beetle_min, lookalike_min)`. Leave empty to use the best ones found.

In [ ]:
# ======================= EDIT THESE =======================

TUNING_DIR = None             # the "tuning" folder made by Step 3b; None -> pick with a pop-up

LABEL_SAMPLE_SIZE = 300       # crops to label by hand

# joint mode: BEETLE_MIN and LOOKALIKE_MIN
JOINT_BEETLE_MIN_VALUES = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]
JOINT_LOOKALIKE_MIN_VALUES = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 0.99]

# separate mode: SEP_INSECT_MIN, SEP_BEETLE_MIN and SEP_LOOKALIKE_MIN
SEP_INSECT_MIN_VALUES = [0.30, 0.50, 0.70]
SEP_BEETLE_MIN_VALUES = [0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
SEP_LOOKALIKE_MIN_VALUES = [0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

TARGET_SENSITIVITY = 0.97     # keep at least 97% of true beetles...
TARGET_SPECIFICITY = 0.90     # ...and reject at least 90% of non-beetles
MAX_REVIEW_PERCENT = 10       # separate mode: at most this % of boxes may go to review

SORT_COMBINATIONS = []        # e.g. [("joint", 0.30, 0.85), ("separate", 0.50, 0.85, 0.50)]; empty -> the best ones found

RANDOM_SEED = 42

# ==========================================================

## Step 2: Load the scores

Reads `candidate_scores.csv` (made by Step 3b) and defines the two decision rules (the same as in Segment_Specimens).

In [ ]:
import os
import shutil
import random
import itertools
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:  # plain Python
    display = print


def find_repo_root(marker='Models'):
    path = os.path.abspath(os.getcwd())
    for _ in range(5):
        if os.path.isdir(os.path.join(path, marker)):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            return os.path.abspath(os.getcwd())
        path = parent
    return os.path.abspath(os.getcwd())


def pick_folder(start_dir):
    import tkinter as tk
    from tkinter import filedialog
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    root.update()
    folder = filedialog.askdirectory(parent=root, title="Choose the 'tuning' folder made by Step 3b of Segment_Specimens",
                                     initialdir=start_dir)
    root.destroy()
    return folder


def decide_joint_array(beetle, lookalike, beetle_min, lookalike_min):
    """Joint rule (same as decide_joint in Segment_Specimens), for whole arrays of scores at once."""
    out = np.where(beetle >= beetle_min, "beetle", "no_insect")
    return np.where((lookalike >= lookalike_min) & (lookalike > beetle), "lookalike", out)


def decide_separate_array(insect, beetle, lookalike, insect_min, beetle_min, lookalike_min):
    """Separate rule (same as decide_separate in Segment_Specimens), for whole arrays of scores at once."""
    beetle_like, lookalike_like = beetle >= beetle_min, lookalike >= lookalike_min
    out = np.where(beetle_like & ~lookalike_like, "beetle",
                   np.where(beetle_like & lookalike_like, "review",
                            np.where(lookalike_like, "lookalike", "neither")))
    return np.where(insect < insect_min, "no_insect", out)


def decisions_for(df, combo):
    """combo is ("joint", beetle_min, lookalike_min) or ("separate", insect_min, beetle_min, lookalike_min)."""
    if combo[0] == "joint":
        return decide_joint_array(df["j_beetle"].to_numpy(), df["j_lookalike"].to_numpy(), combo[1], combo[2])
    return decide_separate_array(df["s_insect"].to_numpy(), df["s_beetle"].to_numpy(), df["s_lookalike"].to_numpy(),
                                 combo[1], combo[2], combo[3])


def names_for(df, combo):
    return df["j_name"] if combo[0] == "joint" else df["s_name"]


def combo_label(combo):
    if combo[0] == "joint":
        return f"joint: BEETLE_MIN = {combo[1]:.2f}, LOOKALIKE_MIN = {combo[2]:.2f}"
    return f"separate: SEP_INSECT_MIN = {combo[1]:.2f}, SEP_BEETLE_MIN = {combo[2]:.2f}, SEP_LOOKALIKE_MIN = {combo[3]:.2f}"


def combo_slug(combo):
    if combo[0] == "joint":
        return f"joint_beetle{combo[1]:.2f}_lookalike{combo[2]:.2f}"
    return f"separate_insect{combo[1]:.2f}_beetle{combo[2]:.2f}_lookalike{combo[3]:.2f}"


REPO_ROOT = find_repo_root()
DATA_DIR = os.path.join(REPO_ROOT, "Data")

if TUNING_DIR is None:
    print("A folder picker has opened. Check your taskbar if you don't see it.", flush=True)
    TUNING_DIR = pick_folder(DATA_DIR)
    if not TUNING_DIR:
        raise RuntimeError("No folder was chosen. Run this cell again and pick the tuning folder.")
if not os.path.isabs(TUNING_DIR):
    TUNING_DIR = os.path.join(DATA_DIR, TUNING_DIR)
TUNING_DIR = os.path.abspath(TUNING_DIR)

scores_path = os.path.join(TUNING_DIR, "candidate_scores.csv")
if not os.path.exists(scores_path):
    raise FileNotFoundError(f"No candidate_scores.csv in {TUNING_DIR}. Run Step 3b of Segment_Specimens.ipynb first.")

scores = pd.read_csv(scores_path, keep_default_na=False)
if "j_beetle" not in scores.columns and "beetle" in scores.columns:  # made by an older Step 3b: joint scores only
    scores = scores.rename(columns={"beetle": "j_beetle", "lookalike": "j_lookalike", "lookalike_name": "j_name", "other": "j_other"})
MODES = ["joint"] + (["separate"] if "s_beetle" in scores.columns else [])
if len(MODES) == 1:
    print("This tuning folder has joint scores only (older Step 3b). Separate mode is skipped; re-run Step 3b to get it.")
scores["crop"] = scores["id"].map(lambda i: os.path.join(TUNING_DIR, "candidates", i + ".jpg"))
scores["family"] = scores["source"].map(lambda s: os.path.basename(os.path.dirname(s)) or "(no subfolder)")

print(f"{len(scores)} scored boxes from {scores['source'].nunique()} photos. Modes available: {', '.join(MODES)}.")
print("Boxes per family:", dict(Counter(scores["family"]).most_common(8)), "...")

## Step 3: Make the labelling sample, then sort it by hand

This copies a random sample of crops into `tuning/labeling/unsorted/`. **Open that folder and move each picture into one of these folders** (next to `unsorted`):

- `beetle`: it is a beetle (any family)
- `lookalike`: a bee, wasp, ant, fly or other non-beetle insect
- `other`: not an insect (label, tape, scale bar, background, a fragment)

Crops too ambiguous to call can be left in `unsorted`; they are ignored. Each file is named after the original photo plus the crop's ID (`<photo name>__c00045.jpg`), so you can find the photo it came from. The names don't show the model's scores, so you are labelling blind. Don't rename the files: the part after `__` is how the notebook knows which crop it is.

To label more later, raise `LABEL_SAMPLE_SIZE` and re-run this cell: it adds new crops and keeps your existing labels. It does nothing if you already have enough.

In [ ]:
LABEL_DIR = os.path.join(TUNING_DIR, "labeling")
TRUTH_CLASSES = ["beetle", "lookalike", "other"]
for name in TRUTH_CLASSES + ["unsorted"]:
    os.makedirs(os.path.join(LABEL_DIR, name), exist_ok=True)

def id_of(filename):
    """The crop id (c00001) is the part after the last "__"; older files are just the id."""
    return os.path.splitext(filename)[0].rsplit("__", 1)[-1]


already = {id_of(f) for name in TRUTH_CLASSES + ["unsorted"] for f in os.listdir(os.path.join(LABEL_DIR, name))}
want = min(LABEL_SAMPLE_SIZE, len(scores))
added = 0
if len(already) < want:
    pool = [i for i in scores["id"] if i not in already]
    random.Random(RANDOM_SEED).shuffle(pool)
    source_of = dict(zip(scores["id"], scores["source"]))
    for cid in pool[:want - len(already)]:
        # name = original photo name + "__" + crop id, e.g. Carabidae_0123__c00045.jpg (several boxes can come from one photo)
        photo = os.path.splitext(os.path.basename(source_of[cid]))[0]
        shutil.copy2(os.path.join(TUNING_DIR, "candidates", cid + ".jpg"), os.path.join(LABEL_DIR, "unsorted", f"{photo}__{cid}.jpg"))
        added += 1

print(f"{added} crops added. Sort the pictures in:\n    {os.path.join(LABEL_DIR, 'unsorted')}")
print("into the folders beetle / lookalike / other next to it, then run the next cell.")

## Step 4: Read your labels

Reads which folder each crop is in and joins it to its scores. The table shows the typical (median) score of each labelled group: if the groups' scores overlap a lot, no threshold will separate them well.

In [ ]:
truth = {}
for name in TRUTH_CLASSES:
    for f in os.listdir(os.path.join(LABEL_DIR, name)):
        if f.lower().endswith(".jpg"):
            truth[id_of(f)] = name
labels = pd.DataFrame({"id": list(truth), "truth": list(truth.values())})
labeled = scores.merge(labels, on="id")

counts = labeled["truth"].value_counts()
n_unsorted = len(os.listdir(os.path.join(LABEL_DIR, "unsorted")))
print(f"{len(labeled)} labelled crops: " + ", ".join(f"{counts.get(n, 0)} {n}" for n in TRUTH_CLASSES) + f"   ({n_unsorted} still unsorted)")

if counts.get("beetle", 0) == 0 or counts.get("lookalike", 0) + counts.get("other", 0) == 0:
    raise RuntimeError("Need at least one labelled beetle and one labelled non-beetle (lookalike or other).")
for name in ("beetle", "lookalike"):
    if counts.get(name, 0) < 30:
        print(f"WARNING: only {counts.get(name, 0)} labelled '{name}' crops; the numbers below will be rough for that group. Label more.")

score_columns = [c for c in ["j_beetle", "j_lookalike", "j_other", "s_insect", "s_beetle", "s_lookalike"] if c in labeled.columns]
display(labeled.groupby("truth")[score_columns].median().round(2))

## Step 5: Try every threshold combination, in both modes

For every combination, applies the rule to your labelled crops and counts:

- **true beetles rejected**: beetles thrown out (lost images)
- **non-beetles passed**: bees, labels etc. that were kept (contamination)
- **review** (separate mode): how many true beetles and non-beetles were set aside for you, and `review_percent`, the share of all boxes that need a human look
- **sensitivity** (share of true beetles kept) and **specificity** (share of non-beetles rejected), counting review boxes as correctly sorted by you afterwards
- **Youden's J** = sensitivity + specificity − 1. A perfect filter scores 1; one no better than guessing scores 0. This is the measure from the [Youden index page](https://metricgate.com/docs/youden-index/) that Tim suggested.

Two ways of picking a winner are shown for each mode, using only combinations that send at most `MAX_REVIEW_PERCENT` of boxes to review:
- **A: best Youden's J.** Treats a lost beetle and a kept bee as equally bad.
- **B: highest specificity while keeping at least `TARGET_SENSITIVITY` of true beetles.** Use this if losing real beetles is the bigger worry (as it is now).

The **verdict** then says whether each mode reaches both targets. The lab's order is to try joint first, and separate only if joint falls short. With a small sample several combinations are practically tied, so prefer one in the middle of a good region of the maps over a lone best cell. The table is also saved as `threshold_results.csv` for sharing.

In [ ]:
is_beetle = (labeled["truth"] == "beetle").to_numpy()
n_beetle, n_other = int(is_beetle.sum()), int((~is_beetle).sum())

combos_to_test = [("joint", b, l) for b, l in itertools.product(JOINT_BEETLE_MIN_VALUES, JOINT_LOOKALIKE_MIN_VALUES)]
if "separate" in MODES:
    combos_to_test += [("separate", i, b, l) for i, b, l in
                       itertools.product(SEP_INSECT_MIN_VALUES, SEP_BEETLE_MIN_VALUES, SEP_LOOKALIKE_MIN_VALUES)]

rows = []
for combo in combos_to_test:
    decision = decisions_for(labeled, combo)
    passed, review = decision == "beetle", decision == "review"
    rejected = ~passed & ~review
    lost, to_review = int((rejected & is_beetle).sum()), int((review & is_beetle).sum())
    leaked = int((passed & ~is_beetle).sum())
    sens, spec = 1 - lost / n_beetle, 1 - leaked / n_other      # review boxes count as sorted correctly by you
    rows.append({"mode": combo[0],
                 "insect_min": combo[1] if combo[0] == "separate" else np.nan,
                 "beetle_min": combo[-2], "lookalike_min": combo[-1],
                 "true_beetles_rejected": lost, "beetles_in_review": to_review,
                 "nonbeetles_passed": leaked, "nonbeetles_in_review": int((review & ~is_beetle).sum()),
                 "review_percent": round(100 * float(review.mean()), 1),
                 "sensitivity": round(sens, 4), "strict_sensitivity": round(1 - (lost + to_review) / n_beetle, 4),
                 "specificity": round(spec, 4), "youden": round(sens + spec - 1, 4)})
results = pd.DataFrame(rows)
results["eligible"] = results["review_percent"] <= MAX_REVIEW_PERCENT
results.to_csv(os.path.join(TUNING_DIR, "threshold_results.csv"), index=False)


def combo_of(row):
    if row["mode"] == "joint":
        return ("joint", round(float(row["beetle_min"]), 2), round(float(row["lookalike_min"]), 2))
    return ("separate", round(float(row["insect_min"]), 2), round(float(row["beetle_min"]), 2), round(float(row["lookalike_min"]), 2))


def describe(row):
    review = f", {int(row['beetles_in_review'])} true beetles + {int(row['nonbeetles_in_review'])} non-beetles to review ({row['review_percent']}% of boxes)" if row["mode"] == "separate" else ""
    return (f"{combo_label(combo_of(row))}\n        {int(row['true_beetles_rejected'])} of {n_beetle} true beetles rejected, "
            f"{int(row['nonbeetles_passed'])} of {n_other} non-beetles passed{review}\n"
            f"        sensitivity {row['sensitivity']:.3f}, specificity {row['specificity']:.3f}, Youden {row['youden']:.3f}")


best_a, best_b = {}, {}
for mode in MODES:
    pool = results[(results["mode"] == mode) & results["eligible"]]
    best_a[mode] = pool.sort_values(["youden", "sensitivity"], ascending=False).iloc[0] if len(pool) else None
    ok = pool[pool["sensitivity"] >= TARGET_SENSITIVITY]
    best_b[mode] = ok.sort_values(["specificity", "sensitivity", "review_percent"], ascending=[False, False, True]).iloc[0] if len(ok) else None

for mode in MODES:
    print(f"=== {mode.upper()} MODE ===")
    print("A (best Youden's J):\n    ", describe(best_a[mode]) if best_a[mode] is not None else "no eligible combination")
    print(f"B (keep >= {TARGET_SENSITIVITY:.0%} of true beetles, then best specificity):\n    ",
          describe(best_b[mode]) if best_b[mode] is not None else f"no combination keeps {TARGET_SENSITIVITY:.0%} of true beetles")
    print()

print("VERDICT (targets: sensitivity >= %.0f%%, specificity >= %.0f%%)" % (100 * TARGET_SENSITIVITY, 100 * TARGET_SPECIFICITY))
for mode in MODES:
    r = best_b[mode]
    if r is None:
        print(f"  {mode}: NO combination keeps {TARGET_SENSITIVITY:.0%} of true beetles.")
    elif r["specificity"] >= TARGET_SPECIFICITY:
        print(f"  {mode}: MEETS the targets (sensitivity {r['sensitivity']:.3f}, specificity {r['specificity']:.3f}).")
    else:
        print(f"  {mode}: does NOT meet the specificity target (best {r['specificity']:.3f} while keeping {TARGET_SENSITIVITY:.0%} of beetles).")

In [ ]:
cmap = plt.get_cmap("viridis").copy()
cmap.set_bad("lightgray")  # combinations that need too much review


def draw(ax, data, title, marks, fmt=None):
    grid = data.pivot(index="beetle_min", columns="lookalike_min", values="value")
    im = ax.imshow(np.ma.masked_invalid(grid.to_numpy()), origin="lower", aspect="auto", cmap=cmap)
    ax.set_xticks(range(len(grid.columns)))
    ax.set_xticklabels([f"{v:.2f}" for v in grid.columns], rotation=90, fontsize=7)
    ax.set_yticks(range(len(grid.index)))
    ax.set_yticklabels([f"{v:.2f}" for v in grid.index], fontsize=7)
    ax.set_xlabel("LOOKALIKE_MIN" if "joint" in title.lower() else "SEP_LOOKALIKE_MIN")
    ax.set_ylabel("BEETLE_MIN" if "joint" in title.lower() else "SEP_BEETLE_MIN")
    ax.set_title(title, fontsize=9)
    ax.figure.colorbar(im, ax=ax, fraction=0.046)
    for row, marker, label in marks:
        if row is not None and float(row["lookalike_min"]) in list(grid.columns) and float(row["beetle_min"]) in list(grid.index):
            ax.plot(list(grid.columns).index(float(row["lookalike_min"])), list(grid.index).index(float(row["beetle_min"])),
                    marker, color="red", markersize=11, markerfacecolor="none", label=label)


# Joint mode
joint = results[results["mode"] == "joint"]
fig, axes = plt.subplots(1, 3, figsize=(18, 4.6))
for ax, col, title in zip(axes, ["sensitivity", "specificity", "youden"],
                          ["Joint: sensitivity (share of true beetles kept)", "Joint: specificity (share of non-beetles rejected)", "Joint: Youden's J"]):
    draw(ax, joint.assign(value=joint[col]), title, [(best_a["joint"], "*", "A"), (best_b["joint"], "s", "B")])
axes[0].legend(title="best", fontsize=8)
plt.tight_layout()
plt.show()

# Separate mode: one pair of maps per SEP_INSECT_MIN
if "separate" in MODES:
    sep = results[results["mode"] == "separate"]
    insect_values = sorted(sep["insect_min"].unique())
    fig, axes = plt.subplots(2, len(insect_values), figsize=(6 * len(insect_values), 8), squeeze=False)
    for k, im_ in enumerate(insect_values):
        part = sep[sep["insect_min"] == im_]
        marks = [(r, m, lab) for r, m, lab in ((best_a["separate"], "*", "A"), (best_b["separate"], "s", "B"))
                 if r is not None and abs(float(r["insect_min"]) - im_) < 1e-9]
        draw(axes[0][k], part.assign(value=part["youden"].where(part["eligible"])), f"Separate, SEP_INSECT_MIN = {im_:.2f}: Youden's J (grey = too much review)", marks)
        draw(axes[1][k], part.assign(value=part["review_percent"]), f"Separate, SEP_INSECT_MIN = {im_:.2f}: % of boxes needing review", [])
    plt.tight_layout()
    plt.show()

for mode in MODES:
    print(f"\nTop 10 for {mode} mode by Youden's J (eligible combinations only):")
    cols = ["insect_min", "beetle_min", "lookalike_min", "true_beetles_rejected", "beetles_in_review", "nonbeetles_passed",
            "nonbeetles_in_review", "review_percent", "sensitivity", "strict_sensitivity", "specificity", "youden"]
    if mode == "joint":
        cols = [c for c in cols if c != "insect_min"]
    top = results[(results["mode"] == mode) & results["eligible"]].sort_values(["youden", "sensitivity"], ascending=False).head(10)
    display(top[cols].reset_index(drop=True))

## Step 6: Sort every scored crop into folders

For each combination chosen, sorts **all** the scored crops (not just the ones you labelled) into

```
tuning/sorted/<combination>/
    1_no_insect_reject/          not an insect (a label, tape...)
    1b_neither_reject/           separate mode: not a beetle and not a look-alike
    2_lookalike_reject/<name>/   rejected as a bee, wasp, etc. (named by the best-matching look-alike)
    3_beetle_pass/               kept as beetles
    4_needs_review/              separate mode: scored as both a beetle and a look-alike
    errors/true_beetles_rejected/   (labelled crops only) beetles that were thrown out, with where they went
    errors/nonbeetles_passed/       (labelled crops only) non-beetles that were kept, with what they really are
```

Flip through `errors/` first: it shows which kinds of beetle are being lost and which look-alikes get through. These folders are only for checking and can be deleted once the thresholds are settled; re-running this cell rebuilds them.

In [ ]:
combos = [tuple(c) for c in SORT_COMBINATIONS]
if not combos:
    for mode in MODES:
        for best in (best_a[mode], best_b[mode]):
            if best is not None:
                combos.append(combo_of(best))
combos = list(dict.fromkeys(combos))  # drop repeats, keep order

truth_by_id = dict(zip(labeled["id"], labeled["truth"]))
FOLDERS = {"no_insect": "1_no_insect_reject", "neither": "1b_neither_reject", "beetle": "3_beetle_pass", "review": "4_needs_review"}


def sort_into_folders(combo):
    base = os.path.join(TUNING_DIR, "sorted", combo_slug(combo))
    if os.path.isdir(base):
        shutil.rmtree(base)
    decisions = decisions_for(scores, combo)
    n = Counter()
    for row, d, name in zip(scores.itertuples(), decisions, names_for(scores, combo)):
        folder = os.path.join("2_lookalike_reject", name) if d == "lookalike" else FOLDERS[d]
        n[folder.split(os.sep)[0]] += 1
        os.makedirs(os.path.join(base, folder), exist_ok=True)
        shutil.copy2(row.crop, os.path.join(base, folder, row.id + ".jpg"))

        t = truth_by_id.get(row.id)
        error = None
        if t == "beetle" and d not in ("beetle", "review"):
            error = ("true_beetles_rejected", f"{row.id}__{d}.jpg")
        elif t is not None and t != "beetle" and d == "beetle":
            error = ("nonbeetles_passed", f"{row.id}__{t}.jpg")
        if error:
            n["errors/" + error[0]] += 1
            os.makedirs(os.path.join(base, "errors", error[0]), exist_ok=True)
            shutil.copy2(row.crop, os.path.join(base, "errors", error[0], error[1]))
    return base, n


for combo in combos:
    base, n = sort_into_folders(combo)
    print(f"{combo_label(combo)}\n    -> {base}")
    for folder in sorted(n):
        print(f"    {folder}: {n[folder]}")

## Step 7: Rejection rate by family

For the first combination above, shows how many boxes in each family were kept, sent to review or rejected, using **all** scored crops (labelled or not). Most boxes in a family should be beetles, so a family with a high rejection rate (Staphylinidae, for example) is a warning that real beetles are being lost, even if your labelled sample was too small to show it. Look at that family's crops in `errors/` and in the reject folders.

In [ ]:
combo = combos[0]
decision = pd.Series(decisions_for(scores, combo), index=scores.index)
table = pd.crosstab(scores["family"], decision).reindex(columns=["beetle", "review", "lookalike", "neither", "no_insect"], fill_value=0)
table["boxes"] = table.sum(axis=1)
table["% rejected"] = (100 * (table["lookalike"] + table["neither"] + table["no_insect"]) / table["boxes"]).round(1)
print(combo_label(combo))
display(table.sort_values("% rejected", ascending=False))

print("\nWhen you have decided, set these in Step 1 of Segment_Specimens.ipynb:")
for combo in combos:
    if combo[0] == "joint":
        print(f'    SCORING_MODE = "joint"\n    BEETLE_MIN = {combo[1]:.2f}\n    LOOKALIKE_MIN = {combo[2]:.2f}\n')
    else:
        print(f'    SCORING_MODE = "separate"\n    SEP_INSECT_MIN = {combo[1]:.2f}\n    SEP_BEETLE_MIN = {combo[2]:.2f}\n    SEP_LOOKALIKE_MIN = {combo[3]:.2f}\n')